# 3회차 · Custom Tool 개발과 AI Agent 구현

**환경:** 로컬에서도 Tool 함수는 테스트 가능, Qwen3-8B 판단/추론은 RunPod GPU 필요.  
**실습 흐름:** Tool 작성 → Registry 등록 → JSON 결정 검증 → Qwen 모델 → Agent 실행 → 로그 저장.  
**이번 개선:** `data_summary(values)` 추가, 안전한 수식 계산, JSON 검증 강화. 4~5회차 서비스에 동일한 Core 함수를 사용합니다.

## STEP 1. RunPod GPU·커널 확인

In [1]:
import sys,torch
print("Python:",sys.executable)
print("CUDA:",torch.cuda.is_available())
if torch.cuda.is_available():print("GPU:",torch.cuda.get_device_name(0))

Python: /usr/local/bin/python
CUDA: True
GPU: NVIDIA A100 80GB PCIe


## STEP 2. 패키지 설치 — RunPod에서 실행하는 경우

In [ ]:
%pip install -U "transformers>=4.53,<5" accelerate pandas sentencepiece

## STEP 3. Custom Tool 4개 구현
AST를 사용하여 Python `eval()` 없이 계산기를 만듭니다. 숫자 목록 통계 Tool도 추가합니다.

In [2]:
"""수업용 Custom Tool: 허용된 연산/명령만 실행한다."""
import ast
import operator as op
import math
from datetime import datetime
from zoneinfo import ZoneInfo

MAX_INPUT = 100
MAX_ABS_RESULT = 1_000_000_000_000
OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
       ast.Div: op.truediv, ast.FloorDiv: op.floordiv, ast.Mod: op.mod,
       ast.USub: op.neg, ast.UAdd: op.pos}

def _calculate(node, depth=0):
    if depth > 15:
        raise ValueError("수식의 깊이가 제한을 초과했습니다.")
    if isinstance(node, ast.Expression):
        return _calculate(node.body, depth+1)
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        value = node.value
    elif isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
        value = OPS[type(node.op)](_calculate(node.operand, depth+1))
    elif isinstance(node, ast.BinOp) and type(node.op) in OPS:
        a = _calculate(node.left, depth+1)
        b = _calculate(node.right, depth+1)
        value = OPS[type(node.op)](a, b)
    elif isinstance(node, ast.BinOp) and isinstance(node.op, ast.Pow):
        a = _calculate(node.left, depth+1)
        b = _calculate(node.right, depth+1)
        if abs(a) > 1_000_000 or abs(b) > 8:
            raise ValueError("거듭제곱 범위를 초과했습니다.")
        value = a ** b
    else:
        raise ValueError("허용되지 않은 수식입니다.")
    if not isinstance(value, (int,float,complex)) or isinstance(value, complex):
        raise ValueError("실수 결과만 허용합니다.")
    if not math.isfinite(value) or abs(value) > MAX_ABS_RESULT:
        raise ValueError("결과 범위를 초과했습니다.")
    return value

def calculator(expression: str) -> dict:
    """사칙연산·거듭제곱 계산. Python eval() 사용 금지."""
    try:
        if not isinstance(expression, str) or len(expression) > MAX_INPUT:
            raise ValueError("수식은 길이 100 이하의 문자열이어야 합니다.")
        tree = ast.parse(expression, mode="eval")
        return {"ok": True, "result": _calculate(tree)}
    except (ValueError, SyntaxError, ArithmeticError, TypeError) as e:
        return {"ok": False, "error": str(e)}

def text_length(text: str) -> dict:
    if not isinstance(text, str):
        return {"ok": False, "error": "text는 문자열이어야 합니다."}
    return {"ok": True, "result": len(text)}

def current_time() -> dict:
    return {"ok": True, "result": datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y-%m-%d %H:%M:%S KST")}

def data_summary(values: list) -> dict:
    """수치 목록 개수/합계/평균/최솟값/최댓값 요약."""
    if not isinstance(values, list) or not (1 <= len(values) <= 100):
        return {"ok": False, "error": "values는 1~100개의 숫자 목록이어야 합니다."}
    if any(type(v) not in (int,float) or not math.isfinite(v) or abs(v)>1e9 for v in values):
        return {"ok": False, "error": "유효한 숫자만 사용할 수 있습니다."}
    return {"ok": True, "result": {"count":len(values), "sum":round(sum(values),4),
            "average":round(sum(values)/len(values),4), "min":min(values), "max":max(values)}}

TOOL_REGISTRY = {"calculator": calculator, "text_length": text_length,
                 "current_time": current_time, "data_summary": data_summary}
TOOL_DESCRIPTIONS = {
    "calculator": {"description":"수식을 정확히 계산", "arguments":{"expression":"계산할 수식 문자열"}},
    "text_length": {"description":"문자열 글자 수 계산", "arguments":{"text":"문자열"}},
    "current_time": {"description":"현재 한국 날짜·시각", "arguments":{}},
    "data_summary": {"description":"숫자 목록의 합계·평균·최솟값·최댓값", "arguments":{"values":"숫자 목록, 예: [10,20,30]"}},
}

def execute_tool(tool_name: str, arguments: dict) -> dict:
    if tool_name not in TOOL_REGISTRY:
        return {"ok": False, "error": f"등록되지 않은 Tool: {tool_name}"}
    if not isinstance(arguments, dict):
        return {"ok": False, "error": "arguments는 JSON 객체(dict)여야 합니다."}
    try:
        return TOOL_REGISTRY[tool_name](**arguments)
    except (TypeError, ValueError) as e:
        return {"ok": False, "error": f"인자 오류: {e}"}

## STEP 4. Tool 정상·오류 입력 테스트

In [3]:
tests=[
  ("계산", "calculator", {"expression":"33*30"}),
  ("길이", "text_length", {"text":"AI Agent"}),
  ("한국시간", "current_time", {}),
  ("데이터 요약", "data_summary", {"values":[10,20,30]}),
  ("등록되지 않은 Tool", "weather", {}),
  ("위험 입력 차단", "calculator", {"expression":"__import__('os').system('id')"}),
]
for label,name,arguments in tests:
    print(label,execute_tool(name,arguments))

계산 {'ok': True, 'result': 990}
길이 {'ok': True, 'result': 8}
한국시간 {'ok': True, 'result': '2026-11-22 19:27:18 KST'}
데이터 요약 {'ok': True, 'result': {'count': 3, 'sum': 60, 'average': 20.0, 'min': 10, 'max': 30}}
등록되지 않은 Tool {'ok': False, 'error': '등록되지 않은 Tool: weather'}
위험 입력 차단 {'ok': False, 'error': '허용되지 않은 수식입니다.'}


## STEP 5. Agent JSON 규약과 공통 함수 정의
`action=tool`이면 Tool을 호출하고, `action=answer`이면 직접 답합니다.

In [4]:
"""LLM의 JSON 결정 → 허용된 Tool 실행 → 최종 응답."""
import json
import re
import time

TOOL_SYSTEM_PROMPT = (
    "당신은 도구 선택 AI Agent다. 가능한 행동은 tool 또는 answer뿐이다.\n"
    "필요한 경우 calculator, text_length, current_time, data_summary 중 하나를 선택한다.\n"
    "숫자 목록 통계에는 data_summary를, 계산에는 calculator를 사용한다.\n"
    "도구가 필요 없는 개념 설명은 answer로 반환한다.\n"
    "반드시 Markdown 없는 JSON 객체 하나만 반환한다.\n"
    '도구 호출: {"action":"tool","tool_name":"calculator","arguments":{"expression":"2+2"}}\n'
    '일반 답변: {"action":"answer","answer":"간단한 설명"}\n'
    + "사용 가능한 Tool: " + json.dumps(TOOL_DESCRIPTIONS, ensure_ascii=False)
)

def parse_agent_json(raw: str) -> dict:
    text = raw.strip()
    # Markdown code fence가 있으면 제거해 파싱을 시도하지만, 실패는 그대로 기록한다.
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
        text = re.sub(r"\s*```$", "", text)
    try:
        obj = json.loads(text)
    except json.JSONDecodeError as e:
        return {"action":"error", "error":f"JSON 파싱 실패: {e}"}
    if not isinstance(obj, dict) or obj.get("action") not in ("tool","answer"):
        return {"action":"error", "error":"action은 tool 또는 answer여야 합니다."}
    if obj["action"] == "tool" and (not isinstance(obj.get("tool_name"),str) or not isinstance(obj.get("arguments"),dict)):
        return {"action":"error", "error":"tool_name/arguments 형식 오류"}
    if obj["action"] == "answer" and not isinstance(obj.get("answer"),str):
        return {"action":"error", "error":"answer는 문자열이어야 합니다."}
    return obj

def make_final_answer(query: str, tool_name: str, tool_result: dict) -> str:
    return generate_llm([
        {"role":"system", "content":"아래 Tool 결과에 근거해 짧고 정확하게 한국어로 답하라."},
        {"role":"user", "content":f"질문: {query}\nTool: {tool_name}\n결과: {json.dumps(tool_result,ensure_ascii=False)}"}
    ], max_new_tokens=160)

def run_agent(user_query: str) -> dict:
    start = time.perf_counter()
    raw = generate_llm([
        {"role":"system", "content":TOOL_SYSTEM_PROMPT},
        {"role":"user", "content":user_query}
    ])
    decision = parse_agent_json(raw)
    result = {"status":"error", "used_tool":False, "tool_name":None,
              "arguments":{}, "tool_result":None, "decision":decision,
              "final_answer":"Agent 결정을 처리하지 못했습니다."}
    if decision["action"] == "answer":
        result.update(status="success", final_answer=decision["answer"])
    elif decision["action"] == "tool":
        tool_name = decision["tool_name"]
        arguments = decision["arguments"]
        tool_result = execute_tool(tool_name, arguments)
        result.update(used_tool=True, tool_name=tool_name,
                      arguments=arguments, tool_result=tool_result)
        if tool_result.get("ok"):
            result.update(status="success",
                          final_answer=make_final_answer(user_query, tool_name, tool_result))
        else:
            result.update(status="tool_error",
                          final_answer=f"Tool 실행 오류: {tool_result.get('error')}")
    result["elapsed_ms"] = round((time.perf_counter()-start)*1000, 1)
    return result

## STEP 6. 구조화 JSON 파싱 단위 테스트

In [5]:
samples=[
  '{"action":"tool","tool_name":"calculator","arguments":{"expression":"10*10"}}',
  '```json\n{"action":"answer","answer":"설명"}\n```',
  '{잘못된 JSON',
  '{"action":"tool","tool_name":"calculator"}'
]
for raw in samples:
    print(parse_agent_json(raw))

{'action': 'tool', 'tool_name': 'calculator', 'arguments': {'expression': '10*10'}}
{'action': 'answer', 'answer': '설명'}
{'action': 'error', 'error': 'JSON 파싱 실패: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)'}
{'action': 'error', 'error': 'tool_name/arguments 형식 오류'}


## STEP 7. Qwen3-8B 모델 로딩 — RunPod GPU 필수
1회차와 같은 `Qwen/Qwen3-8B`, BF16, Thinking OFF입니다.

In [6]:
import os
from pathlib import Path
os.environ.setdefault("HF_HOME", "/workspace/hf-cache")
Path(os.environ["HF_HOME"]).mkdir(parents=True,exist_ok=True)
from transformers import AutoTokenizer, AutoModelForCausalLM
assert torch.cuda.is_available(), "이 단계부터 RunPod GPU 연결이 필요합니다."
MODEL_ID="Qwen/Qwen3-8B"
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,dtype=torch.bfloat16,device_map="auto").eval()
print("모델 로드 완료:",MODEL_ID)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

모델 로드 완료: Qwen/Qwen3-8B


## STEP 8. Hugging Face 로컬 모델 호출 함수

In [7]:
def generate_llm(messages: list, max_new_tokens: int=256) -> str:
    prompt=tokenizer.apply_chat_template(messages,tokenize=False,
                 add_generation_prompt=True,enable_thinking=False)
    inputs=tokenizer(prompt,return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs=model.generate(**inputs,max_new_tokens=max_new_tokens,
                     do_sample=False,pad_token_id=tokenizer.eos_token_id)
    n=inputs["input_ids"].shape[-1]
    return tokenizer.decode(outputs[0][n:],skip_special_tokens=True).strip()

## STEP 9. Agent 판단(Tool 선택) 확인

In [8]:
test_queries=[
 "125 곱하기 8에 30을 더하면?",
 "'AI Agent'의 글자 수를 계산해줘.",
 "지금 한국 날짜와 시간은?",
 "판매액 [10,20,30]의 평균과 합계를 알려줘.",
 "LLM과 AI Agent의 차이를 한 문장으로 설명해줘."
]
for q in test_queries:
    raw=generate_llm([{"role":"system","content":TOOL_SYSTEM_PROMPT},
                      {"role":"user","content":q}])
    print("질문:",q,"\n결정:",parse_agent_json(raw),"\n")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


질문: 125 곱하기 8에 30을 더하면? 
결정: {'action': 'tool', 'tool_name': 'calculator', 'arguments': {'expression': '125*8+30'}} 

질문: 'AI Agent'의 글자 수를 계산해줘. 
결정: {'action': 'tool', 'tool_name': 'text_length', 'arguments': {'text': 'AI Agent'}} 

질문: 지금 한국 날짜와 시간은? 
결정: {'action': 'error', 'error': 'JSON 파싱 실패: Unterminated string starting at: line 1 column 45 (char 44)'} 

질문: 판매액 [10,20,30]의 평균과 합계를 알려줘. 
결정: {'action': 'tool', 'tool_name': 'data_summary', 'arguments': {'values': [10, 20, 30]}} 

질문: LLM과 AI Agent의 차이를 한 문장으로 설명해줘. 
결정: {'action': 'answer', 'answer': 'LLM은 대규모 언어 모델로 텍스트 생성 및 이해에 특화되어 있지만, AI Agent는 특정 작업을 수행하기 위한 목표 지향적 시스템이다.'} 



## STEP 10. Agent 전체 실행 및 결과 로그

In [9]:
import pandas as pd
from IPython.display import display

agent_results=[]
for query in test_queries:
    record=run_agent(query)
    record["question"]=query
    agent_results.append(record)
    print(query,"→",record["status"],record["tool_name"],record["final_answer"])
agent_df=pd.DataFrame(agent_results)
display(agent_df[["question","status","tool_name","elapsed_ms","final_answer"]])

125 곱하기 8에 30을 더하면? → success calculator 125 곱하기 8에 30을 더하면 1030이다.
'AI Agent'의 글자 수를 계산해줘. → success text_length 'AI Agent'의 글자 수는 8입니다.
지금 한국 날짜와 시간은? → error None Agent 결정을 처리하지 못했습니다.
판매액 [10,20,30]의 평균과 합계를 알려줘. → success data_summary 평균은 20.0이고, 합계는 60입니다.
LLM과 AI Agent의 차이를 한 문장으로 설명해줘. → success None LLM은 대규모 언어 모델로 텍스트 생성 및 이해에 특화되어 있지만, AI Agent는 특정 작업을 수행하기 위한 목표 지향적 시스템이다.


,question,status,tool_name,elapsed_ms,final_answer
0,125 곱하기 8에 30을 더하면?,success,calculator,1633.8,125 곱하기 8에 30을 더하면 1030이다.
1,'AI Agent'의 글자 수를 계산해줘.,success,text_length,1114.0,'AI Agent'의 글자 수는 8입니다.
2,지금 한국 날짜와 시간은?,error,NaN,551.2,Agent 결정을 처리하지 못했습니다.
3,"판매액 [10,20,30]의 평균과 합계를 알려줘.",success,data_summary,1504.6,"평균은 20.0이고, 합계는 60입니다."
4,LLM과 AI Agent의 차이를 한 문장으로 설명해줘.,success,NaN,1791.4,"LLM은 대규모 언어 모델로 텍스트 생성 및 이해에 특화되어 있지만, AI Agen..."


## STEP 11. 결과 저장 및 GPU 정리

In [10]:
from pathlib import Path
import gc
output_dir=Path("/workspace/results")
output_dir.mkdir(parents=True,exist_ok=True)
out=output_dir/"03_agent_tool_execution_log.csv"
agent_df.to_csv(out,index=False,encoding="utf-8-sig")
print("저장:",out)
del model,tokenizer
gc.collect();torch.cuda.empty_cache()

저장: /workspace/results/03_agent_tool_execution_log.csv


## STEP 12. 팀 미션 / 다음 차시 연결
- `data_summary`에 `[12, 17, 19, 25]`를 넣어 결과를 해석합니다.
- Tool 설명을 바꾸면 LLM 선택이 달라지는지 비교합니다.
- 존재하지 않는 도구/잘못된 입력이 안전하게 차단되는지 확인합니다.
- **4회차**는 동일한 `tools.py`, `llm.py`, `agent.py`를 FastAPI에 연결합니다.
- 출력: `03_agent_tool_execution_log.csv`와 코드 실습 결과.